# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Chạy lần lượt từ trên xuống. Adapter, `run.json` và evidence được chép lên Google Drive (`MyDrive/lab_day10`) nên mất session thì chạy lại cell, các thí nghiệm đã xong sẽ được bỏ qua.

## 0. Môi trường (Colab T4)

In [ ]:
# Colab: Runtime → T4 GPU, clone repo của bạn rồi %cd vào. Kaggle: gắn dataset code + dataset road-damage, bật GPU + Internet.
import os, shutil
from pathlib import Path
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:  # code repo nằm trong dataset (read-only) → chép ra chỗ ghi được
    shutil.copytree(next(Path("/kaggle/input").rglob("labcommon.py")).parents[1], "/tmp/repo", dirs_exist_ok=True)
    os.chdir("/tmp/repo")
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!pip -q uninstall -y torchao  # image Kaggle có torchao 0.10 → peft 0.21 báo ImportError khi tạo LoRA; lab không dùng torchao

In [ ]:
# Colab: tải dữ liệu bằng Kaggle CLI qua Colab Secrets. KHÔNG viết khoá vào notebook.
if not ON_KAGGLE:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
import os, json, math, random, re, time, gc, shutil, subprocess
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image
from scripts.labcommon import CLASSES, read_jsonl, write_jsonl
from scripts.score import parse_output, iou_xyxy, evaluate

DATA, WORK, FIG = Path("data"), Path("work"), Path("report/figures")
WORK.mkdir(exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
if ON_KAGGLE:  # /kaggle/working là output của kernel
    CKPT = Path("/kaggle/working/ckpt")
else:
    try:  # adapter + evidence được chép lên Drive: session free có thể bị ngắt
        from google.colab import drive
        drive.mount("/content/drive")
        CKPT = Path("/content/drive/MyDrive/lab_day10")
    except ImportError:
        CKPT = WORK / "ckpt"
CKPT.mkdir(parents=True, exist_ok=True)

## 1. Dữ liệu: tải và chia split

In [ ]:
# Kaggle: dataset road-damage đã gắn sẵn trong /kaggle/input
src_arg = f"--source {next(Path('/kaggle/input').rglob('labels-YOLO')).parent}" if ON_KAGGLE else "--kaggle"
!python scripts/prepare_data.py {src_arg}
!python -m pytest -q

In [ ]:
splits = {s: read_jsonl(DATA / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}
for s, rows in splits.items():
    boxes = {c: sum(o["label"] == c for r in rows for o in r["objects"]) for c in CLASSES}
    print(f"{s:5s} {len(rows):4d} ảnh  {boxes}")
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sess["train"] & sess["val"] or sess["train"] & sess["test"] or sess["val"] & sess["test"])
print("Không session nào nằm ở 2 split ✔")

COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, e in zip(axes.flat, random.Random(0).sample([r for r in splits["train"] if r["objects"]], 6)):
    W, H = e["width"], e["height"]
    ax.imshow(Image.open(DATA / e["image_path"])); ax.axis("off"); ax.set_title(e["image_id"], fontsize=8)
    for o in e["objects"]:  # vẽ thẳng từ YOLO (pixel), độc lập với yolo_to_box ở bước 2
        xc, yc, w, h = o["yolo"]
        ax.add_patch(plt.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H,
                                   fill=False, lw=2, color=COLORS[o["label"]]))
plt.tight_layout(); plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
def yolo_to_box(label, yolo, coord="norm1000", width=640, height=360):
    xc, yc, w, h = yolo
    sx, sy = (1000, 1000) if coord == "norm1000" else (width, height)
    box = [(xc - w / 2) * sx, (yc - h / 2) * sy, (xc + w / 2) * sx, (yc + h / 2) * sy]
    return {"label": label, "bbox_2d": [min(max(round(v), 0), m) for v, m in zip(box, (sx, sy, sx, sy))]}


def objects_to_answer(objects, coord="norm1000", width=640, height=360):
    boxes = [yolo_to_box(o["label"], o["yolo"], coord, width, height) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))  # trên→dưới, trái→phải
    return json.dumps(boxes, separators=(",", ":"))


def make_prompt(coord="norm1000"):
    scale = ("integers on a 0-1000 scale relative to the image width and height "
             "(0,0 is top-left, 1000,1000 is bottom-right)" if coord == "norm1000"
             else "integer pixel coordinates of the 640x360 image (x in 0-640, y in 0-360)")
    return ("Detect all road damage in this dashcam image. Classes: pothole, crack, manhole. "
            'Return ONLY a JSON list like [{"label": "pothole", "bbox_2d": [x1, y1, x2, y2]}] '
            f"where bbox_2d coordinates are {scale}. Return [] if there is no damage. No other text.")


PROMPT = make_prompt()


def draw(ax, entry, gt=(), pred=(), title=""):
    # gt/pred: list {"label","bbox"} thang 0–1000 (định dạng của parse_output). GT nét đứt xanh lá, pred màu theo lớp.
    W, H = entry["width"], entry["height"]
    ax.imshow(Image.open(DATA / entry["image_path"])); ax.axis("off"); ax.set_title(title, fontsize=8)
    for boxes, is_gt in ((gt, True), (pred, False)):
        for b in boxes:
            x1, y1, x2, y2 = b["bbox"]
            col = "lime" if is_gt else COLORS.get(b["label"], "magenta")
            ax.add_patch(plt.Rectangle((x1 * W / 1000, y1 * H / 1000), (x2 - x1) * W / 1000, (y2 - y1) * H / 1000,
                                       fill=False, lw=2, color=col, ls="--" if is_gt else "-"))
            ax.text(x1 * W / 1000, y1 * H / 1000 - 3, b["label"], color=col, fontsize=7)

In [ ]:
# Kiểm tra: JSON → (parser của trọng tài) → pixel phải trùng box YOLO gốc (sai số làm tròn ≤ 1 px)
e = next(r for r in splits["train"] if len(r["objects"]) >= 3)
ans = objects_to_answer(e["objects"]); print(ans)
back = parse_output(ans)
for b in back:
    px = [b["bbox"][0] * 0.64, b["bbox"][1] * 0.36, b["bbox"][2] * 0.64, b["bbox"][3] * 0.36]
    assert any(o["label"] == b["label"] and max(abs(p - q) for p, q in zip(px, [
        (o["yolo"][0] - o["yolo"][2] / 2) * 640, (o["yolo"][1] - o["yolo"][3] / 2) * 360,
        (o["yolo"][0] + o["yolo"][2] / 2) * 640, (o["yolo"][1] + o["yolo"][3] / 2) * 360])) <= 1
        for o in e["objects"]), b
print("round-trip ✔")
bad = [r["image_id"] for r in splits["train"] if parse_output(objects_to_answer(r["objects"])) is None]
print("nhãn train bị parser từ chối (box suy biến sau làm tròn):", len(bad))
fig, ax = plt.subplots(figsize=(9, 5)); draw(ax, e, gt=back, title="JSON 0–1000 vẽ ngược"); plt.show()
print(PROMPT)

## 3. E0: zero-shot

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration


def load_base():
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                           bnb_4bit_use_double_quant=True,
                           # transformers 5 so khớp tên từ gốc: "visual" KHÔNG khớp "model.visual.merger..." → ViT bị 4-bit
                           llm_int8_skip_modules=["model.visual", "lm_head"])
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=q, dtype=torch.float16,
                                                            device_map={"": 0})
    fc = model.model.visual.merger.linear_fc1
    assert type(fc) is torch.nn.Linear and fc.weight.dtype == torch.float16, f"projector bị lượng tử hoá: {type(fc)}"
    return model


processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"


def user_msg(entry, prompt):
    img = Image.open(DATA / entry["image_path"]).convert("RGB")
    return {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}


@torch.no_grad()
def predict(model, entries, out_path, prompt=PROMPT, bs=4, max_new_tokens=256):
    model.eval()
    rows = []
    for i in range(0, len(entries), bs):
        batch = entries[i:i + bs]
        inputs = processor.apply_chat_template([[user_msg(e, prompt)] for e in batch], tokenize=True,
                                               add_generation_prompt=True, return_dict=True,
                                               return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        texts = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        rows += [{"image_id": e["image_id"], "raw": t} for e, t in zip(batch, texts)]
        print(f"\r{len(rows)}/{len(entries)}", end="")
    write_jsonl(out_path, rows)
    print(f"\nparse hợp lệ: {sum(parse_output(r['raw']) is not None for r in rows) / len(rows):.3f}")
    return rows


def score(name, pred, run=None, coord="norm1000", split="test"):
    out = (Path("report/evidence") if split == "test" else WORK) / f"{name}.json"
    cmd = ["python", "scripts/score.py", "--pred", str(pred), "--name", name.upper(), "--out", str(out),
           "--split", split, "--coord", coord]
    for k in ("trainable_params", "peak_vram_gb", "train_minutes") if run else ():
        cmd += ["--meta", f"{k}={run[k]}"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout, r.stderr); r.check_returncode()
    (CKPT / "evidence").mkdir(exist_ok=True); shutil.copy(out, CKPT / "evidence")
    return json.loads(out.read_text(encoding="utf-8"))

In [ ]:
model = load_base()
# Gỡ lỗi nhanh trước: predict(model, splits["val"][:40], WORK / "dbg.jsonl")
e0 = predict(model, splits["test"], WORK / "e0.jsonl")
for r in e0[:5]:
    print(repr(r["raw"][:300]))

In [ ]:
score("e0", WORK / "e0.jsonl")
del model; gc.collect(); torch.cuda.empty_cache()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
IGNORE = -100


def encode_example(entry, prompt=PROMPT, coord="norm1000"):
    user = user_msg(entry, prompt)
    answer = objects_to_answer(entry["objects"], coord, entry["width"], entry["height"])
    asst = {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    kw = dict(tokenize=True, return_dict=True, return_tensors="pt")
    p = processor.apply_chat_template([user], add_generation_prompt=True, **kw)
    f = processor.apply_chat_template([user, asst], add_generation_prompt=False, **kw)
    n = p["input_ids"].shape[1]
    assert torch.equal(f["input_ids"][0, :n], p["input_ids"][0]), "prompt không phải tiền tố của chuỗi đầy đủ"
    f["labels"] = f["input_ids"].clone()
    f["labels"][:, :n] = IGNORE  # chỉ học phần trả lời
    return f


def collate(batch):
    pads = {"input_ids": processor.tokenizer.pad_token_id, "attention_mask": 0, "mm_token_type_ids": 0,
            "labels": IGNORE}
    L = max(b["input_ids"].shape[1] for b in batch)
    out = {k: torch.cat([torch.nn.functional.pad(b[k], (0, L - b[k].shape[1]), value=v) for b in batch])
           for k, v in pads.items() if k in batch[0]}
    for k in ("pixel_values", "image_grid_thw"):  # nối, không pad
        out[k] = torch.cat([b[k] for b in batch])
    return out


ex = encode_example(splits["train"][0])
n_img = (ex["input_ids"] == processor.tokenizer.convert_tokens_to_ids(processor.image_token)).sum().item()
print("token ảnh:", n_img); assert n_img == 220
print("phần được học:", repr(processor.tokenizer.decode(ex["labels"][ex["labels"] != IGNORE])))
print({k: tuple(v.shape) for k, v in collate([encode_example(e) for e in splits["train"][:2]]).items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model

LLM_RE = r"model\.language_model\.layers\.\d+\.(self_attn\.[qkvo]_proj|mlp\.(gate|up|down)_proj)"
VIT_RE = r"model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.linear_fc[12])"
PROJ_KEYS = ("visual.merger", "visual.deepstack_merger_list")


def apply_setup(model, cfg):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()  # cần cho gradient checkpointing
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    regex = LLM_RE if cfg["targets"] == "llm" else f"({LLM_RE}|{VIT_RE})"
    n_mod = sum(bool(re.fullmatch(regex, n)) for n, _ in model.named_modules())
    assert n_mod == (348 if "vit" in cfg["targets"] else 252), n_mod
    model = get_peft_model(model, LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=0.05,
                                             target_modules=regex))
    if cfg["train_projector"]:
        for n, p in model.named_parameters():
            if any(k in n for k in PROJ_KEYS):
                p.data = p.data.float()
                if hasattr(p, "grad_dtype"):  # torch mới: đổi .data không cập nhật grad_dtype (vẫn fp16)
                    p.grad_dtype = torch.float32
                p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"module LoRA: {n_mod} · tham số huấn luyện: {n_train:,}")
    return model, n_train

## 6. Vòng huấn luyện

In [ ]:
from transformers import get_cosine_schedule_with_warmup

BASE_CFG = dict(r=16, alpha=32, targets="llm", train_projector=False, lr=2e-4, batch_size=2, grad_accum=4,
                max_steps=300, seed=42, train_fraction=1.0, coord="norm1000")
CFGS = {"e1": BASE_CFG,
        "e2": {**BASE_CFG, "train_projector": True},
        "e3": {**BASE_CFG, "train_projector": True, "targets": "llm+vit"}}


def save_ckpt(model, out_dir):
    model.save_pretrained(out_dir)  # peft CHỈ lưu adapter LoRA
    extra = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n}
    if extra:  # projector E2/E3
        torch.save(extra, out_dir / "extra_trainable.pt")


def train(name, cfg):
    out_dir = CKPT / name
    random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    rows = splits["train"]
    if cfg["train_fraction"] < 1:
        rows = random.Random(cfg["seed"]).sample(rows, round(len(rows) * cfg["train_fraction"]))
    prompt = make_prompt(cfg["coord"])

    def batches():
        rng = random.Random(cfg["seed"])
        while True:
            idx = list(range(len(rows))); rng.shuffle(idx)
            for i in range(0, len(idx) - cfg["batch_size"] + 1, cfg["batch_size"]):
                yield collate([encode_example(rows[j], prompt, cfg["coord"]) for j in idx[i:i + cfg["batch_size"]]])

    torch.cuda.reset_peak_memory_stats()
    model, n_train = apply_setup(load_base(), cfg)
    params = [p for p in model.parameters() if p.requires_grad]
    assert all(p.dtype == torch.float32 for p in params), "tham số huấn luyện phải float32 (GradScaler)"
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * cfg["max_steps"]), cfg["max_steps"])
    scaler = torch.amp.GradScaler("cuda")
    it, log, t0 = batches(), [], time.time()
    model.train()
    for step in range(1, cfg["max_steps"] + 1):
        total = 0.0
        for _ in range(cfg["grad_accum"]):
            batch = {k: v.to(model.device) for k, v in next(it).items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss / cfg["grad_accum"]
            if not torch.isfinite(loss):
                raise RuntimeError(f"loss không hữu hạn ở bước {step} (tràn số fp16?)")
            scaler.scale(loss).backward()
            total += loss.item()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        log.append(total)
        if step % 10 == 0:
            print(f"step {step} loss {sum(log[-10:]) / 10:.4f} lr {sched.get_last_lr()[0]:.2e} "
                  f"{(time.time() - t0) / 60:.1f} phút")
        if step % 100 == 0 or step == cfg["max_steps"]:
            save_ckpt(model, out_dir)
    run = {"name": name, "cfg": cfg, "trainable_params": n_train,
           "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
           "train_minutes": round((time.time() - t0) / 60, 1), "loss": log}
    (out_dir / "run.json").write_text(json.dumps(run, indent=1))
    print({k: run[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")})
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return run

In [ ]:
# Chạy thử 2 bước với setup nặng nhất (E3) để bắt lỗi/OOM trước khi chạy thật (~3 phút)
train("smoke", {**CFGS["e3"], "max_steps": 2, "grad_accum": 1})
shutil.rmtree(CKPT / "smoke")

## 7. Suy luận và chấm E1–E3
Mỗi cell `run_experiment` = train (~20–30 phút) + suy luận test + chấm. Chạy từng cell một.

In [ ]:
def load_trained(name):
    model = PeftModel.from_pretrained(load_base(), CKPT / name)
    extra_path = CKPT / name / "extra_trainable.pt"
    if extra_path.exists():  # thiếu bước này thì E2/E3 ≈ E1
        params = dict(model.named_parameters())
        extra = torch.load(extra_path)
        for n, t in extra.items():
            params[n].data.copy_(t)
        print(f"nạp lại {len(extra)} tensor projector")
    return model.eval()


def run_experiment(name, cfg):
    # Chạy lại được sau khi mất session: bỏ qua train nếu đã có run.json trên Drive
    run_path = CKPT / name / "run.json"
    run = json.loads(run_path.read_text()) if run_path.exists() else train(name, cfg)
    pred = WORK / f"{name}.jsonl"
    if not pred.exists():
        model = load_trained(name)
        predict(model, splits["test"], pred, make_prompt(cfg["coord"]))
        del model; gc.collect(); torch.cuda.empty_cache()
    return score(name, pred, run, coord=cfg["coord"])

In [ ]:
run_experiment("e1", CFGS["e1"])

In [ ]:
run_experiment("e2", CFGS["e2"])

In [ ]:
run_experiment("e3", CFGS["e3"])

In [ ]:
# Bảng cho REPORT.md (copy thẳng)
rows = ["| Mã | Tham số huấn luyện | VRAM đỉnh (GB) | Phút | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole |",
        "|---|---|---|---|---|---|---|---|---|---|"]
for p in sorted(Path("report/evidence").glob("e*.json")):
    d = json.loads(p.read_text(encoding="utf-8")); m, pc = d["meta"], d["per_class"]
    rows.append(f"| {d['name']} | {m.get('trainable_params', 0):,} | {m.get('peak_vram_gb', '')} | "
                f"{m.get('train_minutes', '')} | {d['micro']['f1']:.3f} | {d['f1_ci95'][0]:.3f}–{d['f1_ci95'][1]:.3f} | "
                f"{d['fp_per_image']:.2f} | " + " | ".join(f"{pc[c]['f1']:.3f}" for c in CLASSES) + " |")
print("\n".join(rows))

In [ ]:
def show_examples(name, n=3):
    # 3 ảnh tốt nhất + 3 ảnh tệ nhất (theo F1 từng ảnh) → report/figures/<name>_examples.png
    preds = {r["image_id"]: r["raw"] for r in read_jsonl(WORK / f"{name}.jsonl")}
    scored = sorted(((evaluate([e], preds, n_boot=1)["micro"], e) for e in splits["test"] if e["objects"]),
                    key=lambda t: t[0]["f1"])
    fig, axes = plt.subplots(2, n, figsize=(6 * n, 7.5))
    for ax, (m, e) in zip(axes.flat, scored[-n:][::-1] + scored[:n]):
        draw(ax, e, gt=parse_output(objects_to_answer(e["objects"])), pred=parse_output(preds[e["image_id"]]) or [],
             title=f"{e['image_id']}\nF1={m['f1']:.2f} TP={m['tp']} FP={m['fp']} FN={m['fn']}")
    fig.suptitle(f"{name.upper()} · hàng trên: tốt nhất, hàng dưới: tệ nhất · GT nét đứt xanh lá, dự đoán nét liền "
                 f"(đỏ pothole, vàng crack, xanh manhole)")
    plt.tight_layout(); fig.savefig(FIG / f"{name}_examples.png", dpi=110); plt.show()


for name in ("e0", "e1", "e3"):
    show_examples(name)

In [ ]:
# Câu 3: chấm lại dự đoán E1 (thang 0–1000) như thể là pixel → thấy ngay F1 sụp
!python scripts/score.py --pred work/e1.jsonl --name E1_as_pixel --coord pixel --out work/e1_as_pixel.json

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: đổi ĐÚNG MỘT biến so với setup tốt nhất của E1–E3.
# Mặc định chọn coord_scale: norm1000 chính là setup gốc (chấm lại, không train) → chỉ tốn thêm 1 lần train (pixel).
BEST = max(("e1", "e2", "e3"),
           key=lambda n: json.loads(Path(f"report/evidence/{n}.json").read_text(encoding="utf-8"))["micro"]["f1"])
print("setup tốt nhất:", BEST)
best_run = json.loads((CKPT / BEST / "run.json").read_text())
score("e4_coord_scale_norm1000", WORK / f"{BEST}.jsonl", best_run)
run_experiment("e4_coord_scale_pixel", {**CFGS[BEST], "coord": "pixel"})

# Nếu chọn biến khác thì thay 2 dòng trên bằng:
# rank:      run_experiment("e4_rank_4", {**CFGS[BEST], "r": 4, "alpha": 8})
#            run_experiment("e4_rank_64", {**CFGS[BEST], "r": 64, "alpha": 128})
# data_size: run_experiment("e4_data_size_10pct", {**CFGS[BEST], "train_fraction": 0.1})
#            score("e4_data_size_100pct", WORK / f"{BEST}.jsonl", best_run)

In [ ]:
# E5 (tuỳ chọn): reward IoU + test (Python thuần, không cần GPU), viết trước khi chạy GRPO.
def reward(completion, gt_boxes):
    # gt_boxes: list {"label","bbox"} thang 0–1000
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
    pairs = sorted(((iou_xyxy(g["bbox"], p["bbox"]), i, j) for i, g in enumerate(gt_boxes)
                    for j, p in enumerate(pred) if g["label"] == p["label"]), reverse=True)
    used_g, used_p, total = set(), set(), 0.0
    for v, i, j in pairs:
        if v < 0.5:
            break
        if i not in used_g and j not in used_p:
            used_g.add(i); used_p.add(j); total += v
    return 0.2 + 0.8 * total / max(len(gt_boxes), len(pred), 1)


G = [{"label": "pothole", "bbox": (100, 100, 200, 200)}]
assert reward('[{"label":"pothole","bbox_2d":[100,100,200,200]}]', G) == 1.0  # hoàn hảo
assert reward('```json\n[{"label":"pothole"', G) == 0.0                        # JSON hỏng
assert reward('[{"label":"crack","bbox_2d":[100,100,200,200]}]', G) == 0.2     # sai lớp
assert abs(reward('[{"label":"pothole","bbox_2d":[100,100,200,200]},'
                  '{"label":"crack","bbox_2d":[0,0,50,50]}]', G) - 0.6) < 1e-9  # thừa box bị phạt
assert reward("[]", []) == 1.0 and reward("[]", G) == 0.2                       # rỗng
assert reward('[{"label":"pothole","bbox_2d":[0,0,1000,1000]}]', G) == 0.2     # box khổng lồ không ăn điểm
print("reward tests ✔")

In [ ]:
# E5: GRPO từ adapter tốt nhất (E3), reward IoU. trl 1.9.2: ảnh lấy từ cột "image", cột "gt" được chuyển vào reward.
from datasets import Dataset
from peft import PeftModel
from trl import GRPOConfig, GRPOTrainer

E3_DIR = CKPT / "e3" if (CKPT / "e3").exists() else next(  # notebook riêng: lấy từ Output của lần chạy E0–E4
    p.parent for p in Path("/kaggle/input").rglob("adapter_config.json") if p.parent.name == "e3")
tok = processor.tokenizer
# chỉ lấy ảnh có đáp án vừa max_completion_length=128, để reward không phạt oan ảnh nhiều box (JSON bị cắt)
pool = [e for e in splits["train"] if len(tok(objects_to_answer(e["objects"])).input_ids) < 120]
rows = random.Random(0).sample(pool, 160)
print(f"{len(pool)}/{len(splits['train'])} ảnh train có đáp án < 120 token")
ds = Dataset.from_list([{"prompt": [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}],
                         "image": Image.open(DATA / e["image_path"]).convert("RGB"),
                         "gt": objects_to_answer(e["objects"])} for e in rows])


def reward_iou(completions, gt, **kw):
    texts = [c[0]["content"] if isinstance(c, list) else c for c in completions]
    return [reward(t, parse_output(g)) for t, g in zip(texts, gt)]


torch.cuda.reset_peak_memory_stats()
model = PeftModel.from_pretrained(load_base(), E3_DIR, is_trainable=True)
params = dict(model.named_parameters())
for n, t in torch.load(E3_DIR / "extra_trainable.pt").items():
    params[n].data.copy_(t)  # projector của E3, giữ đóng băng (fp16); GRPO chỉ cập nhật LoRA
cfg = GRPOConfig(output_dir=str(CKPT / "e5"), max_steps=60, learning_rate=5e-6, beta=0.0, num_generations=4,
                 max_completion_length=128, per_device_train_batch_size=4, gradient_accumulation_steps=2,
                 fp16=True, bf16=False, logging_steps=5, save_strategy="no", report_to="none", seed=42,
                 gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
trainer = GRPOTrainer(model=model, reward_funcs=reward_iou, args=cfg, train_dataset=ds, processing_class=processor)
for p in model.parameters():  # trl ép LoRA của model 4-bit về bf16; T4 không có bf16 và GradScaler fp16 cần fp32
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
        if hasattr(p, "grad_dtype"):
            p.grad_dtype = torch.float32
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
t0 = time.time()
trainer.train()
run5 = {"trainable_params": n_train, "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
        "train_minutes": round((time.time() - t0) / 60, 1)}
model.save_pretrained(CKPT / "e5")
Path("report/e5_grpo_log.json").write_text(json.dumps({"run": run5, "log_history": trainer.state.log_history}, indent=1))
KEYS = ("step", "reward", "reward_std", "frac_reward_zero_std", "completions/mean_length", "completions/clipped_ratio")
for l in trainer.state.log_history:
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in l.items() if k in KEYS})
print(run5)

In [ ]:
model.get_base_model().config.use_cache = True
predict(model, splits["test"], WORK / "e5.jsonl")
score("e5", WORK / "e5.jsonl", run5)

# Reward hacking? So sánh hành vi E3 (SFT) và E5 (GRPO) trên test
gts = {e["image_id"]: parse_output(objects_to_answer(e["objects"])) for e in splits["test"]}


def behaviour(path):
    rows = read_jsonl(path)
    parsed = [parse_output(r["raw"]) for r in rows]
    boxes = [b for p in parsed if p for b in p]
    return {"rỗng []": sum(p == [] for p in parsed), "JSON hỏng": sum(p is None for p in parsed),
            "box/ảnh": round(len(boxes) / len(rows), 2),
            "box >25% khung": sum((b["bbox"][2] - b["bbox"][0]) * (b["bbox"][3] - b["bbox"][1]) > 250_000 for b in boxes),
            "reward TB": round(sum(reward(r["raw"], gts[r["image_id"]]) for r in rows) / len(rows), 4)}


E3_PRED = WORK / "e3.jsonl" if (WORK / "e3.jsonl").exists() else next(Path("/kaggle/input").rglob("e3.jsonl"))
analysis = {"GT box/ảnh": round(sum(len(e["objects"]) for e in splits["test"]) / len(splits["test"]), 2),
            "E3 (SFT)": behaviour(E3_PRED), "E5 (GRPO)": behaviour(WORK / "e5.jsonl")}
Path("report/e5_analysis.json").write_text(json.dumps(analysis, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(analysis, ensure_ascii=False, indent=1))

In [ ]:
# Cuối buổi: kiểm tra bài nộp (nhớ commit report/evidence và report/figures)
!python scripts/check_submission.py
if ON_KAGGLE:  # đưa kết quả ra output của kernel
    shutil.copytree("report", "/kaggle/working/report", dirs_exist_ok=True)
    for f in WORK.glob("*.json*"):
        shutil.copy(f, CKPT)